In [14]:
import os
import numpy as np
import pandas as pd
from tqdm import tqdm

PROCESSED_PATH = os.path.join("..", "dataset", "processed")

print("Processed path:", os.path.abspath(PROCESSED_PATH))

Processed path: c:\Users\palak\SANKETSETU\dataset\processed


In [15]:
records = []

for category in sorted(os.listdir(PROCESSED_PATH)):
    category_path = os.path.join(PROCESSED_PATH, category)

    if not os.path.isdir(category_path):
        continue

    for sign in sorted(os.listdir(category_path)):
        sign_path = os.path.join(category_path, sign)

        if not os.path.isdir(sign_path):
            continue

        for file in os.listdir(sign_path):
            if file.endswith(".npy"):
                records.append({
                    "Category": category,
                    "Sign": sign,
                    "File": file,
                    "Path": os.path.join(sign_path, file)
                })

dataset_df = pd.DataFrame(records)

print("Total processed videos:", len(dataset_df))
dataset_df.head()

Total processed videos: 3652


,Category,Sign,File,Path
0,Adjectives,1. loud,MVI_5177.npy,..\dataset\processed\Adjectives\1. loud\MVI_51...
1,Adjectives,1. loud,MVI_5178.npy,..\dataset\processed\Adjectives\1. loud\MVI_51...
2,Adjectives,1. loud,MVI_5179.npy,..\dataset\processed\Adjectives\1. loud\MVI_51...
3,Adjectives,1. loud,MVI_5257.npy,..\dataset\processed\Adjectives\1. loud\MVI_52...
4,Adjectives,1. loud,MVI_5258.npy,..\dataset\processed\Adjectives\1. loud\MVI_52...


In [16]:
labels = sorted(dataset_df["Sign"].unique())

label_to_id = {label: idx for idx, label in enumerate(labels)}
id_to_label = {idx: label for label, idx in label_to_id.items()}

dataset_df["Label"] = dataset_df["Sign"].map(label_to_id)

print("Total unique labels:", len(label_to_id))
dataset_df.head()

Total unique labels: 262


,Category,Sign,File,Path,Label
0,Adjectives,1. loud,MVI_5177.npy,..\dataset\processed\Adjectives\1. loud\MVI_51...,2
1,Adjectives,1. loud,MVI_5178.npy,..\dataset\processed\Adjectives\1. loud\MVI_51...,2
2,Adjectives,1. loud,MVI_5179.npy,..\dataset\processed\Adjectives\1. loud\MVI_51...,2
3,Adjectives,1. loud,MVI_5257.npy,..\dataset\processed\Adjectives\1. loud\MVI_52...,2
4,Adjectives,1. loud,MVI_5258.npy,..\dataset\processed\Adjectives\1. loud\MVI_52...,2


In [17]:
lengths = []

for path in tqdm(dataset_df["Path"]):
    arr = np.load(path)
    lengths.append(arr.shape[0])

dataset_df["Frames"] = lengths

print(dataset_df["Frames"].describe())
print("Maximum frames:", dataset_df["Frames"].max())
print("Minimum frames:", dataset_df["Frames"].min())

100%|██████████| 3652/3652 [00:02<00:00, 1614.87it/s]

count    3652.000000
mean       62.965225
std        14.872715
min        33.000000
25%        53.000000
50%        60.000000
75%        70.000000
max       154.000000
Name: Frames, dtype: float64
Maximum frames: 154
Minimum frames: 33


In [18]:
MAX_FRAMES = dataset_df["Frames"].max()

print("Using max sequence length:", MAX_FRAMES)

def pad_sequence(sequence, max_len):
    padded = np.zeros((max_len, 63), dtype=np.float32)

    length = min(sequence.shape[0], max_len)
    padded[:length] = sequence[:length]

    return padded

Using max sequence length: 154


In [30]:
import numpy as np

# -------------------------------
# Temporal Data Augmentation
# -------------------------------

def random_crop(sequence, max_shift=10):
    """Randomly remove a few frames from the start/end."""
    length = len(sequence)

    start = np.random.randint(0, max_shift + 1)
    end = np.random.randint(0, max_shift + 1)

    cropped = sequence[start:max(length - end, start + 1)]
    return cropped


def speed_augment(sequence):
    """Randomly speed up or slow down a sequence."""
    length = len(sequence)

    # Choose a random speed factor
    factor = np.random.choice([0.8, 0.9, 1.1, 1.2])

    indices = np.arange(0, length, factor)
    indices = np.clip(indices.astype(int), 0, length - 1)

    return sequence[indices]


def landmark_jitter(sequence, sigma=0.01):
    """Add tiny Gaussian noise to landmarks."""
    noise = np.random.normal(0, sigma, sequence.shape)
    return sequence + noise


def augment_sequence(sequence):
    """Create ONE augmented version of a sequence."""
    seq = random_crop(sequence)
    seq = speed_augment(seq)
    seq = landmark_jitter(seq)
    seq = pad_sequence(seq, MAX_FRAMES)
    return seq.astype(np.float32)

print("✅ Augmentation functions ready!")

✅ Augmentation functions ready!


In [19]:
# Remove classes with fewer than 8 videos
min_samples = 8

counts = dataset_df["Label"].value_counts()

valid_labels = counts[counts >= min_samples].index

dataset_df = dataset_df[dataset_df["Label"].isin(valid_labels)].reset_index(drop=True)

print("Remaining classes:", dataset_df["Label"].nunique())
print("Remaining videos:", len(dataset_df))
print("Minimum videos per class:", dataset_df["Label"].value_counts().min())

Remaining classes: 210
Remaining videos: 3298
Minimum videos per class: 8


In [20]:
from sklearn.preprocessing import LabelEncoder

# Re-encode remaining labels to 0...209
label_encoder = LabelEncoder()

dataset_df["Label"] = label_encoder.fit_transform(dataset_df["Label"])

print("Classes after re-encoding:", dataset_df["Label"].nunique())
print("Label range:", dataset_df["Label"].min(), "to", dataset_df["Label"].max())

Classes after re-encoding: 210
Label range: 0 to 209


In [31]:
import numpy as np

X_train, y_train = [], []
X_val, y_val = [], []
X_test, y_test = [], []

rng = np.random.default_rng(42)

for label in np.unique(y):
    idx = np.where(y == label)[0]
    rng.shuffle(idx)

    n = len(idx)

    # 80/10/10 split for each class
    train_end = max(6, int(0.8 * n))
    val_end = train_end + max(1, int(0.1 * n))

    X_train.extend(X[idx[:train_end]])
    y_train.extend(y[idx[:train_end]])

    X_val.extend(X[idx[train_end:val_end]])
    y_val.extend(y[idx[train_end:val_end]])

    X_test.extend(X[idx[val_end:]])
    y_test.extend(y[idx[val_end:]])

# Convert back to numpy arrays
X_train = np.array(X_train, dtype=np.float32)
X_val   = np.array(X_val, dtype=np.float32)
X_test  = np.array(X_test, dtype=np.float32)

y_train = np.array(y_train)
y_val   = np.array(y_val)
y_test  = np.array(y_test)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

print("\nTrain labels:", len(np.unique(y_train)))
print("Validation labels:", len(np.unique(y_val)))
print("Test labels:", len(np.unique(y_test)))

# ==========================================
# Augment ONLY the training set
# ==========================================

augmented_X = []
augmented_y = []

for seq, label in zip(X_train, y_train):
    # Original sample
    augmented_X.append(seq)
    augmented_y.append(label)

    # Augmented version 1
    augmented_X.append(augment_sequence(seq))
    augmented_y.append(label)

    # Augmented version 2
    augmented_X.append(augment_sequence(seq))
    augmented_y.append(label)

# Convert back to numpy arrays
X_train = np.array(augmented_X, dtype=np.float32)
y_train = np.array(augmented_y)

# Shuffle training data
perm = np.random.permutation(len(X_train))
X_train = X_train[perm]
y_train = y_train[perm]

print("✅ Training set augmented!")
print("Train shape:", X_train.shape)
print("Validation shape:", X_val.shape)
print("Test shape:", X_test.shape)

Train: (2536, 154, 63)
Validation: (275, 154, 63)
Test: (487, 154, 63)

Train labels: 210
Validation labels: 210
Test labels: 210
✅ Training set augmented!
Train shape: (7608, 154, 63)
Validation shape: (275, 154, 63)
Test shape: (487, 154, 63)


In [33]:
print("X shape:", X.shape)
print("y shape:", y.shape)
print("Classes:", len(np.unique(y)))
print("Label range:", y.min(), "to", y.max())

counts = np.bincount(y)
print("Minimum samples in any class:", counts.min())

X shape: (3298, 154, 63)
y shape: (3298,)
Classes: 210
Label range: 0 to 209
Minimum samples in any class: 8


In [32]:
TRAIN_PATH = os.path.join("..", "dataset", "training")
os.makedirs(TRAIN_PATH, exist_ok=True)

np.save(os.path.join(TRAIN_PATH, "X_train.npy"), X_train)
np.save(os.path.join(TRAIN_PATH, "X_val.npy"), X_val)
np.save(os.path.join(TRAIN_PATH, "X_test.npy"), X_test)

np.save(os.path.join(TRAIN_PATH, "y_train.npy"), y_train)
np.save(os.path.join(TRAIN_PATH, "y_val.npy"), y_val)
np.save(os.path.join(TRAIN_PATH, "y_test.npy"), y_test)

print("✅ Training dataset saved successfully!")
print("Saved at:", os.path.abspath(TRAIN_PATH))

✅ Training dataset saved successfully!
Saved at: c:\Users\palak\SANKETSETU\dataset\training
